# Satellite-Derived Chlorophyll Time Series for Lakes: MODIS NIR/Red Algorithm

## Overview

This notebook extracts chlorophyll-a concentration estimates using a NIR/Red ratio algorithm from MODIS (Terra and Aqua) for specified lake locations.

MODIS (Moderate Resolution Imaging Spectroradiometer) provides daily global coverage in the visible bands at 250, 500, and 1,000 m resolution from two satellites: Terra (morning overpass) and Aqua (afternoon overpass). This notebook derives chlorophyll-a concentrations using a two-band NIR/Red ratio algorithm specifically designed for turbid Case 2 waters.

- **Purpose**: Generate time series of chlorophyll indices from satellite imagery using NIR/Red algorithm
- **Study Areas**: Detroit Lake and Upper Klamath Lake
- **Satellite Sensors**: MODIS-Aqua and MODIS-Terra
- **Algorithm**: Two-band NIR/Red ratio (Band 2 NIR / Band 1 Red)
- **Resolution**: 500 m
- **Output**: CSV files with date-stamped chlorophyll index values

## Algorithm Background

The NIR/Red algorithm is specifically designed for turbid, productive waters (Case 2) where traditional blue-green algorithms fail due to interference from suspended sediments and colored dissolved organic matter (CDOM). The algorithm exploits:

- **Band 1 (Red, 620-670nm)**: Chlorophyll absorption maximum
- **Band 2 (NIR, 841-876nm)**: Baseline correction, minimal CDOM/sediment effects

**Key References:**
- Gitelson et al. (2008): Simple semi-analytical model for remote estimation of chlorophyll-a in turbid waters
- Moses et al. (2009): NIR/Red algorithms for MERIS data in turbid waters
- Binding et al. (2012): MODIS-derived algal turbidity in Lake Erie

In [1]:
"""
Initialize Google Earth Engine (GEE) connection and authentication.

This cell sets up the GEE Python API for accessing satellite imagery collections.
Authentication is required on first use or when credentials expire.
"""

import ee
import math
import pandas as pd

# Authenticate and initialize GEE with your registered project
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE Cloud Project ID

In [2]:
# -----------------------------
# User parameters
# -----------------------------

lakes = [
    dict(
        name='Detroit',
        lon=-122.184, lat=44.711,
        terra_export='Detroit_MODIS_Terra_Chlorophyll_B2_NIR_B1_Red_500m',
        aqua_export='Detroit_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_500m',
        # Set to None to skip Chl computation (export indices only)
        a=None, b=None
    ),
    dict(
        name='UpperKlamath',
        lon=-121.900, lat=42.400,
        terra_export='UKL_MODIS_Terra_Chlorophyll_B2_NIR_B1_Red_500m',
        aqua_export='UKL_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_500m',
        a=None, b=None
    ),
]

start_date = '2011-01-01'
end_date   = '2025-12-31'

# If a lake has a/b=None, Chl-a won't be computed (only indices exported)
DEFAULT_A = None
DEFAULT_B = None

# ROI & masking
ROI_RADIUS_M         = 1000    # offshore sampling radius (m)
SHORELINE_BUFFER_M   = 500     # erode water mask away from land (m)
WATER_OCC_THRESHOLD  = 75      # JRC occurrence threshold (0-100)
GSW_DATASET_ID       = 'JRC/GSW1_4/GlobalSurfaceWater'  # use '...1_3...' if needed

# MODIS collections and bands (NIR/Red Algorithm)
TERRA_COL_ID = 'MODIS/061/MOD09GA'
AQUA_COL_ID  = 'MODIS/061/MYD09GA'
B_RED        = 'sur_refl_b01'  # Band 1: ~645 nm (Red, 500 m)
B_NIR        = 'sur_refl_b02'  # Band 2: ~859 nm (NIR, 500 m)
SR_SCALE     = 1e-4            # scale factor

In [3]:
# -----------------------------
# QA and water masks
# -----------------------------

def mask_mod09(img):
    """
    MOD09/MYD09 QA-based mask using state_1km:
      - Cloud state (bits 0-1): mask cloudy or mixed
      - Cloud shadow (bit 2)
      - Cirrus (bits 8-9)
      - Internal cloud (bit 10)
      - Snow/ice (bit 12)
    """
    qa = img.select('state_1km')
    cloud_state = qa.bitwiseAnd(3)                     # bits 0-1
    cloudy_or_mixed = cloud_state.eq(1).Or(cloud_state.eq(2))
    shadow   = qa.bitwiseAnd(1 << 2).neq(0)            # bit 2
    cirrus   = qa.bitwiseAnd(3 << 8).neq(0)            # bits 8-9
    intcloud = qa.bitwiseAnd(1 << 10).neq(0)           # bit 10
    snowice  = qa.bitwiseAnd(1 << 12).neq(0)           # bit 12

    mask = cloudy_or_mixed.Or(shadow).Or(cirrus).Or(intcloud).Or(snowice).Not()
    return img.updateMask(mask)

def build_water_mask():
    """
    Persistent open-water mask from JRC Global Surface Water 'occurrence'.
    Erode by SHORELINE_BUFFER_M to mitigate shoreline adjacency.
    """
    gsw = ee.Image(GSW_DATASET_ID).select('occurrence')
    water = gsw.gte(WATER_OCC_THRESHOLD)
    water_eroded = water.focal_min(radius=SHORELINE_BUFFER_M, units='meters')
    return water_eroded

WATER_MASK = build_water_mask()

# -----------------------------------------------------------
# NIR/Red algorithm and statistics computation, per image
# -----------------------------------------------------------

def per_image_stats(img, roi_geom, sensor_tag, a_coeff, b_coeff):
    """
    For a single image:
      - Apply QA & water mask
      - Compute NIR/Red ratio and NDCI (Normalized Difference Chlorophyll Index)
      - Optionally compute Chl-a if a_coeff and b_coeff are not None
      - Reduce over ROI (median) and return a Feature with properties
    """
    img = mask_mod09(img).updateMask(WATER_MASK)

    red = img.select(B_RED).multiply(SR_SCALE)
    nir = img.select(B_NIR).multiply(SR_SCALE)

    # Valid pixel mask
    valid = red.gt(0).And(nir.gt(0))
    
    # NIR/Red ratio (primary algorithm for turbid waters)
    nir_red_ratio = nir.divide(red).updateMask(valid).rename('nir_red_ratio')
    
    # NDCI (Normalized Difference Chlorophyll Index)
    ndci = nir.subtract(red).divide(nir.add(red)).updateMask(valid).rename('ndci')
    
    # Log-transformed NIR/Red ratio for potential calibration
    log_nir_red = nir_red_ratio.log10().rename('log_nir_red')

    # Reduce indices over ROI
    nir_red_stats = nir_red_ratio.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=roi_geom,
        scale=500,
        maxPixels=1e9,
        bestEffort=True
    )
    ndci_stats = ndci.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=roi_geom,
        scale=500,
        maxPixels=1e9,
        bestEffort=True
    )
    log_nir_red_stats = log_nir_red.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=roi_geom,
        scale=500,
        maxPixels=1e9,
        bestEffort=True
    )

    props = ee.Dictionary({
        'datetime': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd HH:mm:ss'),
        'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
        'time': ee.Date(img.get('system:time_start')).format('HH:mm:ss'),
        'sensor': sensor_tag,
        'nir_red_ratio': nir_red_stats.get('nir_red_ratio'),
        'ndci': ndci_stats.get('ndci'),
        'log_nir_red': log_nir_red_stats.get('log_nir_red')
    })

    # Optionally compute Chl-a from calibrated relationship
    def add_chl_props(p):
        # Example: log10(Chl-a) = a * log10(NIR/Red) + b
        log10_chl = log_nir_red.multiply(a_coeff).add(b_coeff)
        chl_img = ee.Image(10).pow(log10_chl).rename('chlor_a')
        chl_stats = chl_img.reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=roi_geom,
            scale=500,
            maxPixels=1e9,
            bestEffort=True
        )
        return ee.Dictionary(p).set('chlor_a', chl_stats.get('chlor_a'))

    if (a_coeff is not None) and (b_coeff is not None):
        props = add_chl_props(props)

    return ee.Feature(None, props)

def imagecollection_to_features(col_id, roi_geom, sensor_tag, a_coeff, b_coeff):
    ic = (ee.ImageCollection(col_id)
          .filterDate(start_date, end_date)
          .filterBounds(roi_geom))

    fc = ic.map(lambda img: per_image_stats(img, roi_geom, sensor_tag, a_coeff, b_coeff))
    # Require NIR/Red ratio to exist (i.e., non-null after masking)
    fc = fc.filter(ee.Filter.notNull(['nir_red_ratio']))
    return fc

In [4]:
# --------------------------------------
# Main loop with client-side CSV export
# --------------------------------------

for lake in lakes:
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(ROI_RADIUS_M)

    a = lake.get('a', DEFAULT_A)
    b = lake.get('b', DEFAULT_B)

    terra_fc = imagecollection_to_features(TERRA_COL_ID, roi, 'Terra', a, b)
    aqua_fc  = imagecollection_to_features(AQUA_COL_ID,  roi, 'Aqua',  a, b)

    # Availability
    print(f"{lake['name']} Terra features =", terra_fc.size().getInfo())
    print(f"{lake['name']} Aqua  features =", aqua_fc.size().getInfo())

    # ---------- Client-side export ----------
    # Terra
    terra_rows = terra_fc.getInfo()['features']
    terra_records = [f['properties'] for f in terra_rows]
    terra_df = pd.DataFrame.from_records(terra_records)
    terra_df = terra_df.sort_values('datetime') if 'datetime' in terra_df.columns else terra_df
    terra_df.to_csv(lake['terra_export'] + '.csv', index=False)
    print(f"Exported: {lake['terra_export']}.csv")

    # Aqua
    aqua_rows = aqua_fc.getInfo()['features']
    aqua_records = [f['properties'] for f in aqua_rows]
    aqua_df = pd.DataFrame.from_records(aqua_records)
    aqua_df = aqua_df.sort_values('datetime') if 'datetime' in aqua_df.columns else aqua_df
    aqua_df.to_csv(lake['aqua_export'] + '.csv', index=False)
    print(f"Exported: {lake['aqua_export']}.csv")
    
    print(f"Completed processing for {lake['name']} Lake\n")

print("Processing complete!")
print("\nOutput files contain the following indices:")
print("- nir_red_ratio: NIR/Red ratio (primary algorithm for turbid waters)")
print("- ndci: Normalized Difference Chlorophyll Index")
print("- log_nir_red: Log10-transformed NIR/Red ratio (for calibration)")
print("- chlor_a: Chlorophyll-a concentration (if calibration coefficients provided)")

Detroit Terra features = 1931
Detroit Aqua  features = 2002
Exported: Detroit_MODIS_Terra_Chlorophyll_B2_NIR_B1_Red_500m.csv
Exported: Detroit_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_500m.csv
Completed processing for Detroit Lake

UpperKlamath Terra features = 2498
UpperKlamath Aqua  features = 2395
Exported: UKL_MODIS_Terra_Chlorophyll_B2_NIR_B1_Red_500m.csv
Exported: UKL_MODIS_Aqua_Chlorophyll_B2_NIR_B1_Red_500m.csv
Completed processing for UpperKlamath Lake

Processing complete!

Output files contain the following indices:
- nir_red_ratio: NIR/Red ratio (primary algorithm for turbid waters)
- ndci: Normalized Difference Chlorophyll Index
- log_nir_red: Log10-transformed NIR/Red ratio (for calibration)
- chlor_a: Chlorophyll-a concentration (if calibration coefficients provided)
